In [56]:
import json
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

In [57]:
# =========================
# CONFIG
# =========================
PREDICTIONS_DIR = Path("../data/processed/asr_predictions/")
NER_MANIFEST    = Path("../data/processed/manifests/ner_manifest_mapped_fixed.jsonl")

PREDICTION_FILES = [
    #"grouped/whisperx_small_word_timestamps_predictions.jsonl",
    "grouped/whisperx_large-v3-turbo_word_timestamps_predictions.jsonl",
    #"grouped/whisper_distil-large-v3_word_timestamps_predictions.jsonl",
    #"grouped/whisper_large-v3_word_timestamps_predictions.jsonl",
    #"grouped/whisper_large-v3-turbo_word_timestamps_predictions.jsonl",
    #"grouped/whisper_medium_word_timestamps_predictions.jsonl",
]

CLEAN_NAMES = {
    "whisper:distil-large-v3":  "Whisper Distil Large v3",
    "whisper:large-v3":         "Whisper Large v3",
    "whisper:large-v3-turbo":   "Whisper Large v3 Turbo",
    "whisper:medium":           "Whisper Medium",
    "whisperx:small":           "WhisperX Small",
    "whisperx:large-v3-turbo":  "WhisperX Large v3 Turbo",
}

# Entity types to evaluate
ENTITY_TYPES = {"PER", "LOC", "ORG"}

# Minimum entity text length to consider (filters single-char noise)
MIN_ENTITY_CHARS = 2

# Number of example drops to show per model
N_EXAMPLES = 20

In [58]:
# =========================
# GOLD CLEANING CONFIG
# =========================

ROLE_KEYWORDS = [
    "designer", "manager", "expert", "engineer",
    "developer", "director", "analyst", "architect",
    "marketing", "industrial", "coach", "researcher",
    "user interface", "market research", "marketing team",
    "marketing person", "project leader", "project supervisor",
    "usability", "interface guy", "interface um", "interface",
]

ANNOTATION_ERRORS = {
    "innovative so",
    "good decision",
    "of",
    "veronica",
    "philip",
}

def is_role_title(s):
    s = str(s).lower()
    return any(kw in s for kw in ROLE_KEYWORDS)

def is_spelled_acronym(s):
    return bool(re.match(r"^([a-z]_){2,}$", str(s).lower()))

def strip_possessive(s):
    if pd.isna(s):
        return s
    return re.sub(r"('s|'ve|')$", "", str(s)).strip()


In [59]:
# =========================
# HELPERS
# =========================

def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalise(text: str) -> str:
    """Lowercase and strip punctuation for fuzzy matching."""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def entity_in_text(entity_text: str, hyp_norm: str) -> bool:
    """Check if a normalised entity appears anywhere in the hypothesis."""
    ent_norm = normalise(strip_possessive(entity_text))
    if not ent_norm or len(ent_norm) < MIN_ENTITY_CHARS:
        return False
    return ent_norm in hyp_norm

In [60]:
# =========================
# LOAD + CLEAN NER MANIFEST
# =========================

def load_ner_entities_clean(manifest_path: Path):
    """
    Returns:
      index -> dict: segment_id -> list of entity dicts with cleaned text/label
      stats -> dict with cleaning counts
    """
    index = defaultdict(list)

    stats = {
        "original_total": 0,
        "removed_role_titles": 0,
        "removed_spelled_acronyms": 0,
        "removed_annotation_errors": 0,
        "possessives_normalised": 0,
        "duplicates_removed": 0,
        "final_total": 0,
    }

    seen = set()

    for row in load_jsonl(manifest_path):
        seg_id = row.get("segment_id")

        for ent in row.get("entities", []):
            label = ent.get("std_label") or ent.get("label", "")
            text  = ent.get("text", "")

            if not text or not label:
                continue
            if label not in ENTITY_TYPES:
                continue

            stats["original_total"] += 1

            text_norm = normalise(text)
            text_norm = strip_possessive(text_norm)

            if str(text).endswith("'s"):
                stats["possessives_normalised"] += 1

            if is_role_title(text_norm):
                stats["removed_role_titles"] += 1
                continue

            if is_spelled_acronym(text_norm):
                stats["removed_spelled_acronyms"] += 1
                continue

            if text_norm in ANNOTATION_ERRORS:
                stats["removed_annotation_errors"] += 1
                continue

            if not text_norm or len(text_norm) < MIN_ENTITY_CHARS:
                continue

            key = (seg_id, text_norm, label)
            if key in seen:
                stats["duplicates_removed"] += 1
                continue
            seen.add(key)

            index[seg_id].append({
                "text": text,                 # original surface form
                "text_norm": text_norm,       # cleaned normalized form
                "label": label,
            })

            stats["final_total"] += 1

    return index, stats


In [61]:
# =========================
# LOAD ASR PREDICTIONS
# =========================

def load_predictions(predictions_dir: Path, files: list) -> dict[str, pd.DataFrame]:
    """Returns dict: model_id -> DataFrame with segment_id, reference_text, asr_text."""
    model_dfs = {}
    for filename in files:
        path = predictions_dir / filename
        if not path.exists():
            print(f"[warn] not found: {path}")
            continue
        rows = load_jsonl(path)
        df = pd.DataFrame(rows)
        if "model" not in df.columns:
            df["model"] = path.stem
        model_id = df["model"].iloc[0]
        model_dfs[model_id] = df
        print(f"Loaded {len(df):>6} rows  [{CLEAN_NAMES.get(model_id, model_id)}]")
    return model_dfs


In [62]:
# =========================
# ENTITY PRESERVATION ANALYSIS
# =========================

def analyse_entity_preservation(
    model_dfs: dict[str, pd.DataFrame],
    ner_index: dict[str, list[dict]],
):
    """
    For each model, check which cleaned gold entities appear in the ASR hypothesis.

    Returns:
      overall_df
      per_type_df
      drop_examples
    """
    overall_rows = []
    per_type_rows = []
    drop_examples = {}

    for model_id, df in model_dfs.items():
        model_name = CLEAN_NAMES.get(model_id, model_id)

        seg_to_hyp = {}
        for _, row in df.iterrows():
            seg_id = row.get("segment_id")
            asr    = row.get("asr_text", "") or ""
            asr_norm = normalise(strip_possessive(asr))

            grouped = row.get("grouped_segments")
            if grouped and isinstance(grouped, list):
                for s in grouped:
                    seg_to_hyp[s] = asr_norm
            elif seg_id:
                seg_to_hyp[seg_id] = asr_norm

        counts = defaultdict(lambda: {"preserved": 0, "total": 0})
        drops  = []

        for seg_id, entities in ner_index.items():
            hyp_norm = seg_to_hyp.get(seg_id)

            # Skip segments with no ASR output (e.g., missing audio)
            if hyp_norm is None or not hyp_norm.strip():
                continue

            for ent in entities:
                label = ent["label"]
                text = ent["text"]
                text_norm = ent["text_norm"]

                counts[label]["total"] += 1
                counts["ALL"]["total"] += 1

                if text_norm and text_norm in hyp_norm:
                    counts[label]["preserved"] += 1
                    counts["ALL"]["preserved"] += 1
                else:
                    drops.append({
                        "segment_id": seg_id,
                        "entity": text,
                        "entity_norm": text_norm,
                        "label": label,
                        "hyp_snippet": hyp_norm[:120],
                    })

        all_c = counts["ALL"]
        total = all_c["total"]
        preserved = all_c["preserved"]

        overall_rows.append({
            "model": model_name,
            "total_entities": total,
            "preserved": preserved,
            "dropped": total - preserved,
            "preservation_rate": round(preserved / total, 4) if total else 0.0,
        })

        for label in sorted(ENTITY_TYPES):
            c = counts[label]
            t = c["total"]
            p = c["preserved"]
            per_type_rows.append({
                "model": model_name,
                "entity_type": label,
                "total": t,
                "preserved": p,
                "dropped": t - p,
                "preservation_rate": round(p / t, 4) if t else 0.0,
            })

        drop_examples[model_name] = drops[:N_EXAMPLES]

    overall_df = pd.DataFrame(overall_rows).sort_values(
        "preservation_rate", ascending=False
    ).reset_index(drop=True)

    per_type_df = pd.DataFrame(per_type_rows)

    return overall_df, per_type_df, drop_examples

In [63]:
# =========================
# RUN
# =========================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("Loading + cleaning NER manifest...")
ner_index, clean_stats = load_ner_entities_clean(NER_MANIFEST)

total_segs = len(ner_index)
total_ents = sum(len(v) for v in ner_index.values())

print(f"  segments with entities      : {total_segs}")
print(f"  original gold entities      : {clean_stats['original_total']}")
print(f"  removed role titles         : {clean_stats['removed_role_titles']}")
print(f"  removed spelled acronyms    : {clean_stats['removed_spelled_acronyms']}")
print(f"  removed annotation errors   : {clean_stats['removed_annotation_errors']}")
print(f"  possessives normalised      : {clean_stats['possessives_normalised']}")
print(f"  duplicate entities removed  : {clean_stats['duplicates_removed']}")
print(f"  final cleaned gold entities : {clean_stats['final_total']}")

print("\nLoading ASR predictions...")
model_dfs = load_predictions(PREDICTIONS_DIR, PREDICTION_FILES)

print("\nAnalysing entity preservation...")
overall_df, per_type_df, drop_examples = analyse_entity_preservation(model_dfs, ner_index)

Loading + cleaning NER manifest...
  segments with entities      : 702
  original gold entities      : 1297
  removed role titles         : 376
  removed spelled acronyms    : 60
  removed annotation errors   : 3
  possessives normalised      : 48
  duplicate entities removed  : 30
  final cleaned gold entities : 828

Loading ASR predictions...
Loaded  20000 rows  [WhisperX Large v3 Turbo]

Analysing entity preservation...


In [64]:
# =========================
# RESULTS
# =========================

print("\n" + "=" * 60)
print("OVERALL ENTITY PRESERVATION")
print("=" * 60)
display_overall = overall_df.copy()
display_overall["preservation_rate"] = (display_overall["preservation_rate"] * 100).round(2).astype(str) + "%"
print(display_overall.to_string(index=False))



OVERALL ENTITY PRESERVATION
                  model  total_entities  preserved  dropped preservation_rate
WhisperX Large v3 Turbo             249        192       57            77.11%


In [65]:
print("\n" + "=" * 60)
print("PRESERVATION BY ENTITY TYPE")
print("=" * 60)
pivot = per_type_df.pivot_table(
    index="model", columns="entity_type", values="preservation_rate"
).round(4) * 100
pivot = pivot.round(2)
pivot.columns = [f"{c} %" for c in pivot.columns]
pivot = pivot.loc[overall_df["model"]]
print(pivot.to_string())



PRESERVATION BY ENTITY TYPE
                         LOC %  ORG %  PER %
model                                       
WhisperX Large v3 Turbo  89.58  83.33  72.12


In [66]:
print("\n" + "=" * 60)
print(f"EXAMPLE DROPPED ENTITIES (top {N_EXAMPLES} per model)")
print("=" * 60)
for model_name, drops in drop_examples.items():
    print(f"\n--- {model_name} ---")
    if not drops:
        print("  No dropped entities found.")
    for d in drops:
        print(f'  [{d["label"]}] "{d["entity"]}"  -> norm="{d["entity_norm"]}"')
        print(f'        hyp: "{d["hyp_snippet"]}"')


EXAMPLE DROPPED ENTITIES (top 20 per model)

--- WhisperX Large v3 Turbo ---
  [PER] "Rajan"  -> norm="rajan"
        hyp: "yeah me again raj as a marketing expert as we have decided in the last meeting that i have to find out i m sorry yeah su"
  [PER] "Arlo"  -> norm="arlo"
        hyp: "it can come under a little bit"
  [PER] "Arlo"  -> norm="arlo"
        hyp: "some information about that about what people other people would require teletext teletext option in their remote connec"
  [PER] "Maarika"  -> norm="maarika"
        hyp: "ok so my name is marika where s the pen ok"
  [PER] "Cat's"  -> norm="cat s"
        hyp: "cut cuts"
  [ORG] "Sky"  -> norm="sky"
        hyp: "yeah and that would be really annoying but that s definitely a possibility"
  [LOC] "Wales"  -> norm="wales"
        hyp: "whales whales for example cool okay"
  [PER] "Jen's"  -> norm="jen s"
        hyp: "i think because the meetings were so regular you know it wasn t like we were alone for very long so you did